# Prepare classification data

Parse the validated dataset and create event grouped train, validation and test splits (70/15/15; seed 42).


In [ ]:
from pathlib import Path
import os, sys, csv, json, ast, random
from collections import Counter, defaultdict

print('Python:', sys.version.split()[0])
print('Kernel:', sys.executable)
print('Current directory:', Path.cwd())

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file() and (p / 'data/forensic_atomic.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Run this notebook from inside the forensic_atomic repository.')
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
CSV_PATH = ROOT / 'data/forensic_atomic.csv'
OUT = ROOT / 'models/phi4_mini/classification/baseline/splits'
SEED = 42
print('Selected CSV:', CSV_PATH, '| exists:', CSV_PATH.exists())
assert CSV_PATH.exists(), 'Missing data/forensic_atomic.csv in the repository.'


In [ ]:
RELATIONS = [
    'xIntent', 'xNeed', 'xAttr', 'xEffect', 'xReact', 'xWant',
    'oEffect', 'oReact', 'oWant', 'isAfter', 'HasSubEvent',
    'isBefore', 'HinderedBy', 'Causes', 'xReason', 'isFilledBy'
]
NEEDED = ['event', 'brief_context', 'crime_category', 'crime_subcategory'] + RELATIONS

def parse_relation(value, idx, column):
    s = str(value or '').strip()
    if not s or s.lower() in {'null', 'none', 'nan'}:
        return []
    try:
        parsed = json.loads(s)
    except json.JSONDecodeError:
        try:
            parsed = ast.literal_eval(s)
        except (ValueError, SyntaxError) as e:
            raise ValueError(f'Row {idx}, column {column}: cannot parse {s[:80]!r}') from e
    if not isinstance(parsed, list):
        raise ValueError(f'Row {idx}, column {column}: expected a list')
    return [x.strip() for x in parsed if isinstance(x, str) and x.strip()]

rows = []
with CSV_PATH.open('r', encoding='utf-8-sig', newline='') as f:
    reader = csv.DictReader(f)
    missing = set(NEEDED) - set(reader.fieldnames or [])
    if missing:
        raise ValueError('Missing CSV columns: ' + str(sorted(missing)))
    for i, raw in enumerate(reader, 2):
        row = {c: str(raw.get(c, '') or '').strip() for c in NEEDED[:4]}
        if not row['event'] or not row['crime_category'] or not row['crime_subcategory']:
            continue
        row['original_row_id'] = str(raw.get('original_row_id') or i-1)
        for name in RELATIONS:
            row[name] = parse_relation(raw[name], i, name)
        rows.append(row)

print('Valid rows:', len(rows))
print('Categories:', Counter(r['crime_category'] for r in rows))
print('Category/subcategory pairs:', len({(r['crime_category'], r['crime_subcategory']) for r in rows}))
assert rows, 'Empty dataset'


## Create grouped splits


In [ ]:
def event_key(event):
    return ' '.join(event.casefold().split())

grouped = defaultdict(list)
for row in rows:
    grouped[event_key(row['event'])].append(row)
assert len(grouped) >= 10, 'At least 10 distinct events are required for a meaningful split.'

all_pairs = {(r['crime_category'], r['crime_subcategory']) for r in rows}
expected_test = len(rows)*0.15
expected_val = len(rows)*0.15

best = None
for attempt in range(300):
    keys = list(grouped)
    random.Random(SEED + attempt).shuffle(keys)
    parts = {'train': [], 'val': [], 'test': []}
    counts = {'train': 0, 'val': 0, 'test': 0}
    for k in keys:
        n = len(grouped[k])
        if abs(counts['test'] + n - expected_test) < abs(counts['test'] - expected_test):
            split = 'test'
        elif abs(counts['val'] + n - expected_val) < abs(counts['val'] - expected_val):
            split = 'val'
        else:
            split = 'train'
        parts[split].extend(grouped[k]); counts[split] += n
    train_pairs = {(r['crime_category'], r['crime_subcategory']) for r in parts['train']}
    missing_pairs = all_pairs - train_pairs
    dev = abs(counts['test'] - expected_test) + abs(counts['val'] - expected_val)
    score = (len(missing_pairs), dev)
    if best is None or score < best[0]:
        best = (score, parts, missing_pairs, attempt)
    if score[0] == 0 and dev <= 2:
        break

_, parts, missing_pairs, used_attempt = best
tr, va, te = (parts[s] for s in ('train', 'val', 'test'))
keys_per_split = [{event_key(r['event']) for r in x} for x in (tr, va, te)]
assert all(keys_per_split[i].isdisjoint(keys_per_split[j]) for i in range(3) for j in range(i+1, 3))
print('Seed:', SEED, '| attempt:', used_attempt)
for split, part in parts.items():
    print(f'{split:5}: {len(part):5} rows ({100*len(part)/len(rows):5.2f}%), {len({event_key(r["event"]) for r in part})} events')
print('Label pairs missing from training:', sorted(missing_pairs))
if missing_pairs:
    print('Warning: unseen training label pairs cannot be selected from the model taxonomy.')


In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
def save_jsonl(path, examples):
    with path.open('w', encoding='utf-8') as f:
        for example in examples:
            f.write(json.dumps(example, ensure_ascii=False) + '\n')

for split in ('train', 'val', 'test'):
    save_jsonl(OUT / f'{split}.jsonl', parts[split])

taxonomy = defaultdict(set)
for r in tr:
    taxonomy[r['crime_category']].add(r['crime_subcategory'])
taxonomy = {k: sorted(v) for k,v in sorted(taxonomy.items())}
(OUT / 'taxonomy.json').write_text(json.dumps(taxonomy, ensure_ascii=False, indent=2), encoding='utf-8')
report = {
    'seed': SEED, 'attempt': used_attempt, 'source': str(CSV_PATH.relative_to(ROOT)),
    'n_rows': len(rows), 'train':len(tr), 'val':len(va), 'test':len(te),
    'missing_train_pairs': sorted([list(p) for p in missing_pairs]),
    'method':'grouped by normalized event, approximately 70/15/15'
}
(OUT / 'split_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print('Files created in:', OUT.resolve())
print([x.name for x in sorted(OUT.iterdir())])


In [ ]:
sample = tr[0]
print('Event:', sample['event'])
print('Context:', sample['brief_context'][:250])
print('Labels:', sample['crime_category'], '/', sample['crime_subcategory'])
print('A → input: event + brief_context')
print('C → input: event + brief_context + relations:', len(RELATIONS), 'e.g. xIntent:', sample['xIntent'][:2])
print('Splits are ready. Open 02_train_evaluate.ipynb with a GPU kernel.')
